# Step 2 Lambda Tuning (Single Fixed Split)

Aligned with **`step2_nowcast_mosquito_log.ipynb`** — same data processing pipeline.

- Fixed tuning cutoff: `tune_split_month` (default `2025-01`)
- Validation: last `inner_val_n_who_months` WHO-observed months inside the window
- Grid search: `lambda_year` × `lambda_reg` × `lambda_lag_reg`
- Output: `outputs_lambda_tuning/lambda_tuning_results.csv` + best triplet

In [1]:
# If needed, install dependencies in your environment:
# pip install -U pandas numpy torch matplotlib

# Prevent kernel crashes from OpenMP/MKL library conflicts
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Tuple, List, Optional

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Further limit thread count
torch.set_num_threads(1)

In [2]:
@dataclass
class Step2Config:
    # Inputs — aligned with step2_nowcast_mosquito_log.ipynb
    step1_predictions_path: str = os.path.join('preprocessing_outputs', 'preprocessed_monthly.csv')
    step1_pred_col: str = 'x_pred'

    # Modeling window
    start_month: str = '2021-01'

    # Target / source names
    target_source: str = 'WHO'

    # Exogenous signals
    google_sources: Tuple[str, str] = (
        'Google_Trends_Dengue_fever',
        'Google_Trends_Dengue_vaccine',
    )

    # Wikipedia pageviews — same as mosquito_log
    use_wiki: bool = True
    wiki_transform: str = 'log1p'

    # Mosquito Wikipedia pageviews — same as mosquito_log
    use_mosquito: bool = True
    mosquito_transform: str = 'log1p'

    # Feature engineering — same as mosquito_log
    lags_y: Tuple[int, ...] = (1, 2, 12)
    use_month_dummies: bool = False
    seasonal_period: int = 12
    use_fourier: bool = True
    fourier_K: int = 2

    # OpenDengue yearly proxy sources
    yearly_proxy_sources_priority: Tuple[str, ...] = (
        'OpenDengue_State_Aggregated',
        'OpenDengue_National_Yearly',
    )

    # Loss weights — same defaults as mosquito_log (will be overridden during tuning)
    lambda_who: float = 5.0
    lambda_year: float = 0.01
    lambda_reg: float = 0.1
    lambda_lag_reg: float = 5e-3

    # Target scale and clip — MUST match mosquito_log exactly
    target_scale: float = 1.0   # log1p target; no additional scaling
    clip_nonnegative: bool = False  # expm1 back-transform guarantees positive

    # Optimization (used during tuning inner loop)
    lr: float = 3e-3
    seed: int = 42

    # Overfitting controls
    early_stop: bool = True
    min_delta: float = 1e-7

    # Feature scaling
    standardize_features: bool = True

    # ── Tuning-specific settings ──────────────────────────────────────────
    tune_split_month: str = '2025-01'         # fixed cutoff for tuning window
    inner_val_n_who_months: int = 3            # last N WHO-observed months used as validation
    tune_epochs: int = 6000
    tune_patience: int = 300
    tune_lambda_year_grid: Tuple[float, ...] = (0.0, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0)
    tune_lambda_reg_grid: Tuple[float, ...] = (1e-5, 1e-4, 1e-3, 1e-2, 1e-1)
    tune_lambda_lag_reg_grid: Tuple[float, ...] = (1e-5, 1e-4, 1e-3, 5e-3, 1e-2)

    # Output
    outdir: str = 'outputs_lambda_tuning'


cfg = Step2Config()
cfg

Step2Config(step1_predictions_path='preprocessing_outputs/preprocessed_monthly.csv', step1_pred_col='x_pred', start_month='2021-01', target_source='WHO', google_sources=('Google_Trends_Dengue_fever', 'Google_Trends_Dengue_vaccine'), use_wiki=True, wiki_transform='log1p', use_mosquito=True, mosquito_transform='log1p', lags_y=(1, 2, 12), use_month_dummies=False, seasonal_period=12, use_fourier=True, fourier_K=2, yearly_proxy_sources_priority=('OpenDengue_State_Aggregated', 'OpenDengue_National_Yearly'), lambda_who=5.0, lambda_year=0.01, lambda_reg=0.1, lambda_lag_reg=0.005, target_scale=1.0, clip_nonnegative=False, lr=0.003, seed=42, early_stop=True, min_delta=1e-07, standardize_features=True, tune_split_month='2025-01', inner_val_n_who_months=3, tune_epochs=6000, tune_patience=300, tune_lambda_year_grid=(0.0, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0), tune_lambda_reg_grid=(1e-05, 0.0001, 0.001, 0.01, 0.1), tune_lambda_lag_reg_grid=(1e-05, 0.0001, 0.001, 0.005, 0.01), outdir='outputs_lambda_tuning

In [3]:
def ensure_outdir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def parse_monthly_date_any(s: pd.Series) -> pd.Series:
    x = s.astype(str)
    is_ym = x.str.match('^[0-9]{4}-[0-9]{2}$')
    x = np.where(is_ym, x + '-01', x)
    dt = pd.to_datetime(x, errors='coerce')
    return dt


def load_master_csv() -> pd.DataFrame:
    from load_processed_data import build_master_df
    return build_master_df()


def build_monthly_wide(df: pd.DataFrame) -> pd.DataFrame:
    m = df[df['resolution'].astype(str).str.lower().eq('monthly')].copy()
    m['date'] = parse_monthly_date_any(m['date'])
    m = m.dropna(subset=['date']).copy()
    wide = (
        m.pivot_table(index='date', columns='source', values='value', aggfunc='mean')
        .sort_index()
    )
    return wide


def build_yearly_proxy(df: pd.DataFrame, priority_sources: Tuple[str, ...]) -> pd.DataFrame:
    y = df[df['resolution'].astype(str).str.lower().eq('yearly')].copy()
    y['year'] = pd.to_numeric(y['date'], errors='coerce').astype('Int64')
    y = y.dropna(subset=['year'])
    y['year'] = y['year'].astype(int)

    pivot = (
        y.pivot_table(index='year', columns='source', values='value', aggfunc='mean')
        .sort_index()
    )

    chosen = []
    for year, row in pivot.iterrows():
        val = np.nan
        src = None
        for s in priority_sources:
            if s in row.index and pd.notna(row[s]):
                val = float(row[s])
                src = s
                break
        if pd.notna(val):
            chosen.append((year, val, src))

    return pd.DataFrame(chosen, columns=['year', 'od_total', 'od_source'])


def load_step1_predictions(cfg: Step2Config) -> pd.Series:
    if not os.path.exists(cfg.step1_predictions_path):
        return pd.Series(dtype=float)

    s1 = pd.read_csv(cfg.step1_predictions_path)
    if 'date' not in s1.columns:
        raise ValueError(f"Step 1 predictions file missing 'date': {cfg.step1_predictions_path}")

    s1['date'] = parse_monthly_date_any(s1['date'])
    s1 = s1.dropna(subset=['date']).copy()
    s1 = s1.sort_values('date')

    col = cfg.step1_pred_col if cfg.step1_pred_col in s1.columns else None
    if col is None:
        cand = [c for c in s1.columns if 'pred' in c.lower()]
        if not cand:
            cand = [c for c in s1.columns if c != 'date']
        if not cand:
            raise ValueError('No prediction column found in Step 1 predictions file.')
        col = cand[0]

    ser = s1.set_index('date')[col].astype(float)
    return ser



def load_wiki_monthly(cfg: Step2Config) -> pd.Series:
    """Load monthly Wikipedia dengue pageviews. Returns pd.Series indexed by month (Timestamp at month start)."""
    if not getattr(cfg, 'use_wiki', False):
        return pd.Series(dtype=float)
    from load_processed_data import get_wiki_dengue_df
    w = get_wiki_dengue_df()   # returns [Month, Total_Views]
    w = w.copy()
    w['date'] = parse_monthly_date_any(w['Month'])
    w = w.dropna(subset=['date']).sort_values('date')
    s = pd.to_numeric(w['Total_Views'], errors='coerce')
    return pd.Series(s.values, index=w['date']).groupby(level=0).mean()

def load_mosquito_monthly(cfg: Step2Config) -> pd.DataFrame:
    """Load monthly mosquito Wikipedia pageviews. Returns DataFrame indexed by month (Timestamp at month start)."""
    if not getattr(cfg, 'use_mosquito', False):
        return pd.DataFrame()
    from load_processed_data import get_mosquito_df
    m = get_mosquito_df()   # returns [timestamp, TOTAL_MONTHLY_VIEWS]
    m = m.copy()
    m['date'] = parse_monthly_date_any(m['timestamp'])
    m = m.dropna(subset=['date']).sort_values('date')
    feat_cols = [c for c in m.columns if c not in {'timestamp', 'date'}]
    feat = m[feat_cols].apply(pd.to_numeric, errors='coerce')
    feat['date'] = m['date'].values
    return feat.groupby('date').mean()

def add_month_dummies(df: pd.DataFrame) -> pd.DataFrame:
    dt = pd.to_datetime(df.index)
    m = pd.get_dummies(dt.month, prefix='m', drop_first=True)
    m.index = df.index
    return pd.concat([df, m.astype(float)], axis=1)



def add_fourier_features(df: pd.DataFrame, period: int = 12, K: int = 2, prefix: str = 's') -> pd.DataFrame:
    """Add Fourier seasonality terms based on month-of-year."""
    dt = pd.to_datetime(df.index)
    m = dt.month.astype(float)
    out = df.copy()
    for k in range(1, int(K) + 1):
        out[f'{prefix}_sin{k}'] = np.sin(2 * np.pi * k * m / float(period))
        out[f'{prefix}_cos{k}'] = np.cos(2 * np.pi * k * m / float(period))
    return out



def build_design_matrix(wide: pd.DataFrame, cfg: Step2Config) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray, List[str]]:
    """
    Build design matrix.
    Critical fix: lagged variables x_tilde must be scaled by target_scale to match target scale.
    """
    start_dt = pd.to_datetime(cfg.start_month + '-01')
    wide = wide.loc[wide.index >= start_dt].copy()

    target = cfg.target_source
    if target not in wide.columns:
        wide[target] = np.nan

    g1, g2 = cfg.google_sources
    for g in [g1, g2]:
        if g not in wide.columns:
            raise ValueError(f"Missing Google source '{g}' in monthly data.")

    y_who = pd.to_numeric(wide[target], errors='coerce')  # raw WHO units (log1p applied inside training functions)

    step1_ser = load_step1_predictions(cfg)
    wide['STEP1_est'] = step1_ser.reindex(wide.index)

    # x_tilde: WHO (raw) when observed, else Step1 estimate (for lagged inputs)
    # Use raw (un-transformed) values here, then apply log1p for consistency with target
    y_who_raw = pd.to_numeric(wide[target], errors='coerce')
    x_tilde_raw = y_who_raw.where(y_who_raw.notna(), wide['STEP1_est'])
    x_tilde_scaled = np.log1p(x_tilde_raw)  # log1p to match log1p-transformed target

    X_df = pd.DataFrame(index=wide.index)
    
    # Normalize Google Trends from [0, 100] to [0, 1]
    X_df['g_fever'] = pd.to_numeric(wide[g1], errors='coerce') / 100.0
    X_df['g_vaccine'] = pd.to_numeric(wide[g2], errors='coerce') / 100.0


    # Wikipedia pageviews regressor
    if getattr(cfg, 'use_wiki', False):
        wiki_ser = load_wiki_monthly(cfg)
        wide['WIKI_raw'] = wiki_ser.reindex(wide.index)
        wiki_raw = pd.to_numeric(wide['WIKI_raw'], errors='coerce')
        if str(getattr(cfg, 'wiki_transform', 'log1p')).lower() == 'log1p':
            X_df['wiki_views'] = np.log1p(wiki_raw.astype(float))
        else:
            X_df['wiki_views'] = wiki_raw.astype(float)


    # Mosquito Wikipedia pageviews regressor (external feature)
    if getattr(cfg, 'use_mosquito', False):
        mosq_df = load_mosquito_monthly(cfg)
        if mosq_df is not None and len(mosq_df) > 0:
            mosq_df = mosq_df.reindex(wide.index)
            for c in mosq_df.columns:
                raw = pd.to_numeric(mosq_df[c], errors='coerce')
                out_col = f"mosq_{str(c).lower()}"
                if str(getattr(cfg, 'mosquito_transform', 'log1p')).lower() == 'log1p':
                    X_df[out_col] = np.log1p(raw.astype(float))
                else:
                    X_df[out_col] = raw.astype(float)

    # Use scaled values for lagged variables
    for k in cfg.lags_y:
        X_df[f'x_tilde_lag{k}'] = x_tilde_scaled.shift(k)

    # Seasonality features (periodic time series)
    # Prefer Fourier terms when enabled; otherwise fall back to month dummies.
    if getattr(cfg, 'use_fourier', False):
        X_df = add_fourier_features(
            X_df,
            period=int(getattr(cfg, 'seasonal_period', 12)),
            K=int(getattr(cfg, 'fourier_K', 2)),
            prefix='s'
        )
    elif cfg.use_month_dummies:
        X_df = add_month_dummies(X_df)


    X_df['intercept'] = 1.0

    # Light imputation on exogenous features only (lags can remain NaN and be imputed later)
    
    exo_cols = ['g_fever', 'g_vaccine']
    if 'wiki_views' in X_df.columns:
        exo_cols.append('wiki_views')

    exo_cols += [c for c in X_df.columns if c.startswith('mosq_')]

    for col in exo_cols:
        if col in X_df.columns:
            X_df[col] = X_df[col].interpolate(limit_direction='both').ffill().bfill()

    feature_cols = [c for c in X_df.columns if c != 'intercept'] + ['intercept']
    X = X_df[feature_cols].to_numpy(dtype=np.float32)
    y = y_who.to_numpy(dtype=np.float32)
    mask_who = np.isfinite(y)


    return X_df, X, y, mask_who, feature_cols


def build_year_constraints(dates: pd.DatetimeIndex, yearly_proxy: pd.DataFrame, train_mask: np.ndarray) -> List[tuple]:
    df_dates = pd.DataFrame({'date': pd.to_datetime(dates)})
    df_dates['year'] = df_dates['date'].dt.year
    df_dates['month'] = df_dates['date'].dt.month

    constraints = []
    for _, r in yearly_proxy.iterrows():
        y = int(r['year'])
        od_total = float(r['od_total'])
        od_src = str(r['od_source'])

        idx = df_dates.index[df_dates['year'].eq(y)].to_numpy()
        if len(idx) == 0:
            continue

        # require all 12 months and all months in training window
        months_present = set(df_dates.loc[idx, 'month'].tolist())
        if months_present != set(range(1, 13)):
            continue
        if not np.all(train_mask[idx]):
            continue

        constraints.append((y, idx, od_total, od_src))

    return constraints


def time_series_split_mask(index: pd.DatetimeIndex, split_month: str, horizon_months: int = 2):
    split_dt = pd.to_datetime(split_month + '-01')
    test_end_dt = split_dt + pd.DateOffset(months=int(horizon_months))
    train_mask = index <= split_dt
    test_mask = (index > split_dt) & (index <= test_end_dt)
    return train_mask, test_mask


def standardize_features(X: np.ndarray, train_mask: np.ndarray, skip_cols: List[int] = None):
    """
    Standardize feature matrix.
    skip_cols: column indices to skip standardization (e.g., intercept term).
    """
    X2 = X.copy()
    n, p = X2.shape
    means = np.zeros(p, dtype=np.float32)
    stds = np.ones(p, dtype=np.float32)
    
    if skip_cols is None:
        skip_cols = []

    for j in range(p):
        col = X2[:, j]
        col_train = col[train_mask]
        
        # For intercept or specified columns, skip standardization
        if j in skip_cols:
            # Only impute NaN
            col = np.where(np.isfinite(col), col, 0.0)
            X2[:, j] = col
            continue
        
        m = np.nanmean(col_train)
        s = np.nanstd(col_train)
        if not np.isfinite(m):
            m = 0.0
        if (not np.isfinite(s)) or s <= 1e-12:
            s = 1.0
        means[j] = m
        stds[j] = s

        # impute NaNs with train mean, then standardize
        col = np.where(np.isfinite(col), col, m)
        X2[:, j] = (col - m) / s

    return X2, means, stds


def train_step2_joint_loss(
    X: np.ndarray,
    y_who: np.ndarray,
    mask_who: np.ndarray,
    year_constraints: List[tuple],
    train_mask: np.ndarray,
    cfg: Step2Config,
    feature_cols: List[str],
    ):
    """
    Train with differentiated regularization: stronger penalty on lag coefficients.
    """
    device = torch.device('cpu')

    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    y_scaled = np.log1p(np.maximum(y_who, 0.0))  # log1p transform inside training
    y_t = torch.tensor(y_scaled, dtype=torch.float32, device=device)
    mask_who_t = torch.tensor(mask_who & train_mask, dtype=torch.bool, device=device)

    year_terms = []
    for (year, idx, od_total, od_src) in year_constraints:
        year_terms.append(
            (year, torch.tensor(idx, dtype=torch.long, device=device), float(len(idx) * np.log1p(float(od_total) / len(idx))), od_src)  # corrected: n_months * log1p(od_total/n_months)
        )

    # Identify lag feature indices for differentiated regularization
    lag_indices = [i for i, col in enumerate(feature_cols) if 'lag' in col.lower()]

    p = X_t.shape[1]
    beta = torch.nn.Parameter(torch.zeros(p, dtype=torch.float32, device=device))
    opt = torch.optim.Adam([beta], lr=cfg.lr)

    rows = []
    best_loss = np.inf
    best_beta = None
    bad_epochs = 0

    for epoch in range(1, cfg.epochs + 1):
        opt.zero_grad(set_to_none=True)
        x = X_t @ beta

        if mask_who_t.any():
            diff_who = x[mask_who_t] - y_t[mask_who_t]
            L_who = (diff_who ** 2).mean()
        else:
            L_who = torch.tensor(0.0, device=device)

        if len(year_terms) > 0:
            diffs = []
            for _, idx_t, od_total_scaled, _ in year_terms:
                year_sum = x.index_select(0, idx_t).sum()
                diffs.append((year_sum - od_total_scaled) ** 2)
            L_year = torch.stack(diffs).mean()
        else:
            L_year = torch.tensor(0.0, device=device)

        # Differentiated regularization: base + extra penalty on lag coefficients
        L_reg_base = (beta ** 2).sum()
        if lag_indices:
            beta_lag = beta[lag_indices]
            L_reg_lag = (beta_lag ** 2).sum()
        else:
            L_reg_lag = torch.tensor(0.0, device=device)
        
        L_reg = L_reg_base
        L_total = cfg.lambda_who * L_who + cfg.lambda_year * L_year + cfg.lambda_reg * L_reg + cfg.lambda_lag_reg * L_reg_lag
        L_total.backward()
        opt.step()

        val = float(L_total.detach().cpu().item())
        rows.append({
            'epoch': epoch,
            'L_total': val,
            'L_who': float(L_who.detach().cpu().item()),
            'L_year': float(L_year.detach().cpu().item()),
            'L_reg': float(L_reg.detach().cpu().item()),
            'L_lag_reg': float(L_reg_lag.detach().cpu().item()) if lag_indices else 0.0,
        })

        if not np.isfinite(val):
            raise RuntimeError('Training diverged (loss is NaN/Inf).')

        if cfg.early_stop:
            if val < best_loss - cfg.min_delta:
                best_loss = val
                best_beta = beta.detach().cpu().numpy().copy()
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= cfg.patience:
                    break

    beta_hat = beta.detach().cpu().numpy() if best_beta is None else best_beta
    loss_df = pd.DataFrame(rows)
    return beta_hat, loss_df

In [4]:

def compute_rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.nanmean((y_true - y_pred) ** 2)))

def compute_mse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.nanmean((y_true - y_pred) ** 2))


def safe_mape(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.where(np.abs(y_true) < 1e-12, np.nan, np.abs(y_true))
    return float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100.0)


def make_predictions(X: np.ndarray, beta_hat: np.ndarray, cfg: Step2Config) -> np.ndarray:
    x_log = X @ beta_hat  # prediction in log1p-space
    x = np.expm1(x_log)   # back-transform to WHO units; always > -1
    if cfg.clip_nonnegative:
        x = np.maximum(x, 0.0)
    return x


def plot_loss_curve(loss_df: pd.DataFrame, outdir: Path) -> None:
    fig = plt.figure()
    plt.plot(loss_df['epoch'], loss_df['L_total'], label='Total')
    if (loss_df['L_who'] != 0).any():
        plt.plot(loss_df['epoch'], loss_df['L_who'], label='WHO')
    if (loss_df['L_year'] != 0).any():
        plt.plot(loss_df['epoch'], loss_df['L_year'], label='Yearly')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (log scale)')
    plt.title('Training Loss Curve (Step 2)')
    plt.legend(loc=legend_loc)
    plt.tight_layout()
    fig.savefig(outdir / 'loss_curve_step2.png', dpi=200)
    plt.close(fig)



def plot_who_vs_pred(
    dates: pd.DatetimeIndex,
    y_who: np.ndarray,
    x_pred: np.ndarray,
    train_mask: np.ndarray,
    test_mask: np.ndarray,
    outdir: Path,
    fname: str = 'who_vs_pred_step2.png',
    title: str = None,
    x_step1: Optional[np.ndarray] = None,
    legend_loc: str = 'best',
) -> None:
    fig = plt.figure(figsize=(10, 4))

    display_mask = np.asarray(train_mask, dtype=bool) | np.asarray(test_mask, dtype=bool)
    x_pred_plot = np.asarray(x_pred, dtype=float).copy()
    x_pred_plot[~display_mask] = np.nan

    split_dt = dates[train_mask].max() if np.asarray(train_mask).any() else None
    test_end_dt = dates[test_mask].max() if np.asarray(test_mask).any() else None

    if split_dt is not None and test_end_dt is not None and test_end_dt >= split_dt:
        plt.axvspan(split_dt, test_end_dt, alpha=0.10, color='gray', zorder=0)

    plt.plot(dates, x_pred_plot, label='Predicted (Model)')
    plt.plot(dates, y_who, label='WHO observed')

    # Optional: overlay Step 1 predictions only for historical missing WHO months.
    # Future Step 1 extrapolation (purple line in the old plot) is intentionally removed.
    if x_step1 is not None:
        x_step1 = np.asarray(x_step1, dtype=float)
        y_who_arr = np.asarray(y_who, dtype=float)

        who_idx = np.where(np.isfinite(y_who_arr))[0]
        last_who = int(who_idx.max()) if who_idx.size > 0 else -1

        miss_mask = (~np.isfinite(y_who_arr)) & np.isfinite(x_step1)
        miss_idx = np.where(miss_mask)[0]

        if miss_idx.size > 0:
            runs = []
            s = miss_idx[0]
            prev = miss_idx[0]
            for k in miss_idx[1:]:
                if k == prev + 1:
                    prev = k
                else:
                    runs.append((s, prev))
                    s = prev = k
            runs.append((s, prev))

            first_hist = True

            for (a, b) in runs:
                # keep only historical missing blocks, not future extrapolation blocks
                if a > last_who:
                    continue

                seg_dates = list(dates[a:b + 1])
                seg_vals = list(x_step1[a:b + 1])

                if a - 1 >= 0 and np.isfinite(y_who_arr[a - 1]):
                    seg_dates = [dates[a - 1]] + seg_dates
                    seg_vals = [y_who_arr[a - 1]] + seg_vals

                if b + 1 < len(dates) and np.isfinite(y_who_arr[b + 1]):
                    seg_dates = seg_dates + [dates[b + 1]]
                    seg_vals = seg_vals + [y_who_arr[b + 1]]

                label = 'Data preprocessing (only where WHO is missing)' if first_hist else None
                first_hist = False
                plt.plot(seg_dates, seg_vals, color='#2ecc71', linewidth=2.5, label=label)

    if x_step1 is not None:
        y_who_arr = np.asarray(y_who, dtype=float)
        who_idx = np.where(np.isfinite(y_who_arr))[0]
        if who_idx.size > 0:
            first_who = who_idx.min()
            if first_who > 0 and (~np.isfinite(y_who_arr[:first_who])).any():
                plt.axvline(
                    dates[first_who],
                    linestyle=':',
                    color='black',
                    linewidth=2.0,
                    label='WHO begins'
                )

    if split_dt is not None:
        plt.axvline(split_dt, linestyle='--', color='gray', label='Train/Test split')
    if test_end_dt is not None:
        plt.axvline(test_end_dt, linestyle='--', color='lightgray', label='2-month test end')

    plt.xlabel('Date')
    plt.ylabel('Monthly dengue cases')
    plt.title(title if title is not None else 'WHO vs Prediction (Model)')
    plt.legend()
    plt.tight_layout()
    fig.savefig(outdir / fname, dpi=200)
    plt.close(fig)



def plot_yearly_vs_od(pred_df: pd.DataFrame, yearly_proxy: pd.DataFrame, outdir: Path) -> None:
    pred_year = pred_df.groupby('year', as_index=False)['x_pred'].sum().rename(columns={'x_pred': 'pred_year_total'})
    if yearly_proxy.empty:
        return
    merged = pred_year.merge(yearly_proxy, on='year', how='inner').sort_values('year')

    fig = plt.figure()
    plt.plot(merged['year'], merged['pred_year_total'], marker='o', label='Predicted yearly sum')
    plt.plot(merged['year'], merged['od_total'], marker='o', label='OpenDengue yearly total')
    plt.xlabel('Year')
    plt.ylabel('Total dengue cases (year)')
    plt.title('Yearly Aggregation: Prediction vs OpenDengue (Model)')
    plt.legend()
    plt.tight_layout()
    fig.savefig(outdir / 'yearly_vs_opendengue_step2.png', dpi=200)
    plt.close(fig)

    merged.to_csv(outdir / 'yearly_comparison_step2.csv', index=False)



def plot_step1_vs_step2(pred_df: pd.DataFrame, outdir: Path):
    if 'x_step1' not in pred_df.columns:
        return
    fig = plt.figure(figsize=(10, 4))
    dates = pd.to_datetime(pred_df['date'] + '-01')
    plt.plot(dates, pred_df['x_step1'], label='Data preprocessing')
    plt.plot(dates, pred_df['x_pred'], label='Model')
    plt.xlabel('Date')
    plt.ylabel('Monthly dengue cases')
    plt.title('Data preprocessing vs Model')
    plt.legend()
    plt.tight_layout()
    fig.savefig(outdir / 'step1_vs_step2.png', dpi=200)
    plt.close(fig)


from statistics import NormalDist

def compute_beta_ci_hessian(
    X: np.ndarray,
    y_who: np.ndarray,
    mask_who: np.ndarray,
    year_constraints: List[tuple],
    train_mask: np.ndarray,
    cfg: Step2Config,
    feature_cols: List[str],
    beta_hat: np.ndarray,
    feat_mean: Optional[np.ndarray] = None,
    feat_std: Optional[np.ndarray] = None,
    intercept_idx: Optional[int] = None,
    alpha: Optional[float] = None,
) -> dict:
    """
    Approximate coefficient confidence intervals under a Gaussian noise model
    on WHO-observed training months, using a normal approximation from the
    (quadratic) objective Hessian.

    Notes:
      - Treats yearly totals as deterministic constraints (no noise).
      - Returns CIs for:
          * beta (scaled target units)
          * coef_y_units_per_stdX (WHO units per 1 std of X, if standardized)
          * coef_y_units_per_rawX (WHO units per 1 raw-unit of X, if standardized)
    """
    if alpha is None:
        alpha = float(getattr(cfg, 'beta_ci_alpha', 0.05))

    beta_hat = np.asarray(beta_hat, dtype=float)
    p = beta_hat.shape[0]

    who_train_mask = (mask_who.astype(bool) & train_mask.astype(bool))
    idx = np.where(who_train_mask)[0]
    n_who = int(idx.size)
    if n_who == 0:
        nan = np.full(p, np.nan, dtype=float)
        return dict(
            beta_se=nan, beta_ci_lo=nan, beta_ci_hi=nan,
            coef_std_se=nan, coef_std_ci_lo=nan, coef_std_ci_hi=nan,
            coef_raw_se=nan, coef_raw_ci_lo=nan, coef_raw_ci_hi=nan,
            sigma2=np.nan, df_eff=np.nan,
        )

    Xw = np.asarray(X[idx, :], dtype=float)
    y_scaled = np.log1p(np.maximum(np.asarray(y_who[idx], dtype=float), 0.0))  # log1p inside CI

    # Build Hessian (scaled units, matches train_step2_joint_loss)
    lam_who = float(cfg.lambda_who)
    lam_year = float(cfg.lambda_year)
    lam_reg = float(cfg.lambda_reg)
    lam_lag = float(cfg.lambda_lag_reg)

    # WHO term: (lam_who / n_who) * ||Xw beta - y||^2
    H_core = (lam_who / n_who) * (Xw.T @ Xw)

    # Year constraints term: (lam_year / n_year) * sum_y (c_y^T beta - od_y)^2
    n_year = int(len(year_constraints))
    if n_year > 0 and lam_year != 0.0:
        for (_, idx_year, od_total, _) in year_constraints:
            c = np.asarray(X[np.asarray(idx_year, dtype=int), :], dtype=float).sum(axis=0)
            H_core += (lam_year / n_year) * np.outer(c, c)

    # Regularization terms
    H_core += lam_reg * np.eye(p, dtype=float)

    lag_indices = [i for i, col in enumerate(feature_cols) if 'lag' in str(col).lower()]
    if lag_indices and lam_lag != 0.0:
        D = np.zeros(p, dtype=float)
        D[np.asarray(lag_indices, dtype=int)] = 1.0
        H_core += lam_lag * np.diag(D)

    # Full Hessian of L_total
    Hess = 2.0 * H_core

    # Invert Hessian (small p; ridge-like => should be well-conditioned)
    try:
        Hess_inv = np.linalg.inv(Hess)
    except np.linalg.LinAlgError:
        Hess_inv = np.linalg.pinv(Hess)

    # Effective degrees of freedom for WHO fitted values (approx.)
    XtX = Xw.T @ Xw
    df_eff = (2.0 * lam_who / n_who) * float(np.trace(XtX @ Hess_inv))

    # Residual variance estimate on WHO-observed training months (scaled)
    r = (Xw @ beta_hat) - y_scaled
    RSS = float(np.sum(r ** 2))
    denom = max(n_who - df_eff, 1.0)
    sigma2 = RSS / denom

    # Var(beta_hat) = sigma2 * A A^T, A = Hess_inv * (2 lam_who / n_who) * Xw^T
    cfac = (2.0 * lam_who / n_who)
    Var_beta = sigma2 * (cfac ** 2) * (Hess_inv @ XtX @ Hess_inv)

    se_beta = np.sqrt(np.clip(np.diag(Var_beta), 0.0, np.inf))

    z = float(NormalDist().inv_cdf(1.0 - alpha / 2.0))
    ci_lo = beta_hat - z * se_beta
    ci_hi = beta_hat + z * se_beta

    # Convert to WHO units per standardized X
    coef_std = beta_hat * float(cfg.target_scale)
    coef_std_se = se_beta * float(cfg.target_scale)
    coef_std_ci_lo = ci_lo * float(cfg.target_scale)
    coef_std_ci_hi = ci_hi * float(cfg.target_scale)

    # Convert to WHO units per raw X (requires feat_std/feat_mean if standardized)
    if (feat_std is not None) and (feat_mean is not None):
        feat_std = np.asarray(feat_std, dtype=float)
        feat_mean = np.asarray(feat_mean, dtype=float)

        # Transform covariance into coef_std space
        Var_coef_std = (float(cfg.target_scale) ** 2) * Var_beta

        # Build linear transform T: coef_std -> coef_raw (including intercept adjustment)
        T = np.zeros((p, p), dtype=float)
        for j in range(p):
            T[j, j] = 1.0 / float(feat_std[j]) if float(feat_std[j]) != 0.0 else 0.0

        if intercept_idx is not None:
            # intercept_raw = coef_std[int] - sum_j mean_j * (coef_std[j]/std_j)
            T[intercept_idx, :] = 0.0
            T[intercept_idx, intercept_idx] = 1.0
            for j in range(p):
                if j == intercept_idx:
                    continue
                stdj = float(feat_std[j]) if float(feat_std[j]) != 0.0 else 1.0
                T[intercept_idx, j] = - float(feat_mean[j]) / stdj

        Var_coef_raw = T @ Var_coef_std @ T.T
        se_raw = np.sqrt(np.clip(np.diag(Var_coef_raw), 0.0, np.inf))
        coef_raw = (coef_std / feat_std)
        if intercept_idx is not None:
            coef_raw = coef_raw.copy()
            coef_raw[intercept_idx] = coef_std[intercept_idx] - float(np.dot(coef_raw, feat_mean))

        coef_raw_ci_lo = coef_raw - z * se_raw
        coef_raw_ci_hi = coef_raw + z * se_raw
    else:
        se_raw = np.full(p, np.nan, dtype=float)
        coef_raw = np.full(p, np.nan, dtype=float)
        coef_raw_ci_lo = np.full(p, np.nan, dtype=float)
        coef_raw_ci_hi = np.full(p, np.nan, dtype=float)

    return dict(
        beta_se=se_beta,
        beta_ci_lo=ci_lo,
        beta_ci_hi=ci_hi,
        coef_std_se=coef_std_se,
        coef_std_ci_lo=coef_std_ci_lo,
        coef_std_ci_hi=coef_std_ci_hi,
        coef_raw_se=se_raw,
        coef_raw_ci_lo=coef_raw_ci_lo,
        coef_raw_ci_hi=coef_raw_ci_hi,
        Var_beta=Var_beta,
        z=z,
        sigma2=sigma2,
        df_eff=df_eff,
    )


def compute_yearly_mape_from_constraints(y_pred: np.ndarray, year_constraints: list) -> float:
    """Average yearly MAPE over the provided year_constraints."""
    if not year_constraints:
        return float('nan')
    errs = []
    for (year, idx, od_total, _src) in year_constraints:
        pred_total = float(np.nansum(y_pred[np.asarray(idx, dtype=int)]))
        denom = od_total if abs(od_total) > 1e-12 else np.nan
        errs.append(abs(pred_total - od_total) / denom)
    return float(np.nanmean(errs) * 100.0)


## Load Data & Build Design Matrix

In [5]:
import copy
from pathlib import Path

np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

df = load_master_csv()
wide = build_monthly_wide(df)
yearly_proxy = build_yearly_proxy(df, cfg.yearly_proxy_sources_priority)

X_df, X_raw, y_who, mask_who, feature_cols = build_design_matrix(wide, cfg)

print('Total months:', len(X_df))
print('WHO observed months:', int(mask_who.sum()))
print('Feature count:', X_raw.shape[1])
print('Features:', feature_cols)

Total months: 63
WHO observed months: 26
Feature count: 12
Features: ['g_fever', 'g_vaccine', 'wiki_views', 'mosq_total_monthly_views', 'x_tilde_lag1', 'x_tilde_lag2', 'x_tilde_lag12', 's_sin1', 's_cos1', 's_sin2', 's_cos2', 'intercept']


## Lambda Tuning — Single Fixed Split

**Alignment**: identical data processing to `step2_nowcast_mosquito_log.ipynb`:
- Target: raw WHO (log1p applied inside `train_step2_joint_loss`)
- x_tilde lags: `log1p(x_tilde_raw)` — consistent with log1p target
- Year constraint: `n_months × log1p(od_total / n_months)` in log space
- `make_predictions`: `expm1(X @ beta)` back to WHO units
- `target_scale = 1.0`, `clip_nonnegative = False`

**Split design**:
- Cutoff: `tune_split_month` (default `2025-01`)
- Validation: last `inner_val_n_who_months` WHO-observed months inside the window
- Training inner: everything in the window **before** validation months
- Grid: `lambda_year` × `lambda_reg` × `lambda_lag_reg`
- Selection: minimum validation RMSE on WHO-observed months

In [6]:
# ── Build tuning split ───────────────────────────────────────────────────
def make_inner_train_val_masks(train_mask_full, mask_who, n_val_who_months):
    """Pick validation months as the last n WHO-observed months inside the training window."""
    idx_who = np.where(train_mask_full & mask_who)[0]
    if len(idx_who) <= n_val_who_months:
        raise ValueError(
            f'Not enough WHO-observed months ({len(idx_who)}) to form a validation set of {n_val_who_months}.'
        )
    val_idx = idx_who[-n_val_who_months:]
    val_mask = np.zeros_like(train_mask_full, dtype=bool)
    val_mask[val_idx] = True
    train_inner = train_mask_full & (~val_mask)
    return train_inner, val_mask


TUNE_SPLIT_MONTH = cfg.tune_split_month
train_full_mask, _ = time_series_split_mask(X_df.index, TUNE_SPLIT_MONTH)
train_inner_mask, val_mask = make_inner_train_val_masks(
    train_full_mask, mask_who, cfg.inner_val_n_who_months
)

train_dates = X_df.index[train_inner_mask]
val_dates   = X_df.index[val_mask & mask_who]

print(f'Tuning cutoff:           {TUNE_SPLIT_MONTH}')
print(f'Training months:         {train_dates[0].strftime("%Y-%m")} to {train_dates[-1].strftime("%Y-%m")} ({len(train_dates)} months)')
print(f'Validation months (WHO): {[d.strftime("%Y-%m") for d in val_dates]}')

# Intercept column index — skip standardization
intercept_idx = feature_cols.index('intercept') if 'intercept' in feature_cols else None
skip_cols = [intercept_idx] if intercept_idx is not None else []

# Standardize using TRAIN-INNER only (no leakage into validation)
if cfg.standardize_features:
    X_tune, feat_mean_t, feat_std_t = standardize_features(X_raw, train_inner_mask, skip_cols=skip_cols)
else:
    X_tune = X_raw.copy()

# Year constraints from TRAIN-INNER only
year_constraints_t = build_year_constraints(X_df.index, yearly_proxy, train_inner_mask)
print(f'Year constraints in training: {[t[0] for t in year_constraints_t]}')

Tuning cutoff:           2025-01
Training months:         2021-01 to 2024-10 (46 months)
Validation months (WHO): ['2024-11', '2024-12', '2025-01']
Year constraints in training: [2021, 2022, 2023]


In [7]:
lambda_year_grid    = list(cfg.tune_lambda_year_grid)
lambda_reg_grid     = list(cfg.tune_lambda_reg_grid)
lambda_lag_reg_grid = list(cfg.tune_lambda_lag_reg_grid)
total_combos = len(lambda_year_grid) * len(lambda_reg_grid) * len(lambda_lag_reg_grid)
print(f'Searching {total_combos} combinations '
      f'({len(lambda_year_grid)} × {len(lambda_reg_grid)} × {len(lambda_lag_reg_grid)})...')

tune_rows = []
val_who_mask = val_mask & mask_who

for ly in lambda_year_grid:
    for lr in lambda_reg_grid:
        for llr in lambda_lag_reg_grid:
            cfg_t = copy.deepcopy(cfg)
            cfg_t.lambda_year    = float(ly)
            cfg_t.lambda_reg     = float(lr)
            cfg_t.lambda_lag_reg = float(llr)
            cfg_t.epochs         = int(cfg.tune_epochs)
            cfg_t.patience       = int(cfg.tune_patience)

            beta_hat_t, _ = train_step2_joint_loss(
                X=X_tune,
                y_who=y_who,
                mask_who=mask_who,
                year_constraints=year_constraints_t,
                train_mask=train_inner_mask,
                cfg=cfg_t,
                feature_cols=feature_cols,
            )

            y_pred_t = make_predictions(X_tune, np.asarray(beta_hat_t, dtype=float), cfg_t)

            tune_rows.append({
                'lambda_year':    float(ly),
                'lambda_reg':     float(lr),
                'lambda_lag_reg': float(llr),
                'val_RMSE':       compute_rmse(y_who[val_who_mask], y_pred_t[val_who_mask]),
                'val_MAPE_%':     safe_mape(y_who[val_who_mask], y_pred_t[val_who_mask]),
                'year_MAPE_%':    compute_yearly_mape_from_constraints(y_pred_t, year_constraints_t),
            })

tune_df = pd.DataFrame(tune_rows).sort_values('val_RMSE').reset_index(drop=True)
print('Done.')
tune_df.head(20)

Searching 175 combinations (7 × 5 × 5)...
Done.


,lambda_year,lambda_reg,lambda_lag_reg,val_RMSE,val_MAPE_%,year_MAPE_%
0,0.01,0.01000,0.01000,5198.356370,33.074952,70.301721
1,0.01,0.01000,0.00500,5207.651990,34.324777,66.463087
2,0.01,0.01000,0.00100,5312.490807,35.399943,63.445110
3,0.01,0.01000,0.00010,5350.091844,35.652004,62.772812
4,0.01,0.01000,0.00001,5354.135973,35.677330,62.705921
5,0.01,0.00100,0.00500,5487.316717,36.694027,120.387530
6,0.01,0.00010,0.00500,5517.967759,36.902354,128.917110
7,0.01,0.00001,0.00500,5521.100206,36.922770,129.816839
8,0.01,0.00100,0.01000,5525.848484,35.290209,127.470626
9,0.01,0.00010,0.01000,5569.182115,35.484403,136.551000


In [8]:
best_row = tune_df.iloc[0]
BEST_LAMBDA_YEAR    = float(best_row['lambda_year'])
BEST_LAMBDA_REG     = float(best_row['lambda_reg'])
BEST_LAMBDA_LAG_REG = float(best_row['lambda_lag_reg'])

print('=== Tuning Summary ===')
print(f'  Tuning cutoff:           {TUNE_SPLIT_MONTH}')
print(f'  Validation months (WHO): {[d.strftime("%Y-%m") for d in val_dates]}')
print()
print('=== Best Parameters (min val_RMSE) ===')
print(f'  best_lambda_year    = {BEST_LAMBDA_YEAR}')
print(f'  best_lambda_reg     = {BEST_LAMBDA_REG}')
print(f'  best_lambda_lag_reg = {BEST_LAMBDA_LAG_REG}')
print(f'  val_RMSE            = {float(best_row["val_RMSE"]):.4f}')
print(f'  val_MAPE_%          = {float(best_row["val_MAPE_%"]):.1f}%')
print(f'  year_MAPE_%         = {float(best_row["year_MAPE_%"]):.1f}%')

# Save full tuning table
outdir = Path(cfg.outdir)
outdir.mkdir(parents=True, exist_ok=True)
tune_df.to_csv(outdir / 'lambda_tuning_results.csv', index=False)
print(f'\nSaved tuning results ({len(tune_df)} rows) → {(outdir / "lambda_tuning_results.csv").resolve()}')

=== Tuning Summary ===
  Tuning cutoff:           2025-01
  Validation months (WHO): ['2024-11', '2024-12', '2025-01']

=== Best Parameters (min val_RMSE) ===
  best_lambda_year    = 0.01
  best_lambda_reg     = 0.01
  best_lambda_lag_reg = 0.01
  val_RMSE            = 5198.3564
  val_MAPE_%          = 33.1%
  year_MAPE_%         = 70.3%

Saved tuning results (175 rows) → /Users/cathiesmac/Downloads/Nowcast Dengue in India/outputs_lambda_tuning/lambda_tuning_results.csv
